In [ ]:
import pandas as pd
import pandas_ta as ta
import pyotp
from datetime import datetime, timedelta
import time
import csv
import asyncio
import nest_asyncio
import logging
import websocket as ws_client
import threading
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict, deque
import numba_indicators
from threading import Lock

from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self

import logging
#logging.basicConfig(level=logging.DEBUG)
logging.getLogger('websocket').setLevel(logging.INFO)

usercred = pd.read_excel("usercred.xlsx")
user    = usercred.iloc[0,0]
pwd     = usercred.iloc[0,1]
vc      = usercred.iloc[0,2]
app_key = usercred.iloc[0,3]
imei    = usercred.iloc[0,4]
qr = usercred.iloc[0,5]
factor2 = pyotp.TOTP(qr).now()

api = ShoonyaApiPy()
ret = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)

In [ ]:
# Load data
import pandas as pd

fno_scrips_mcx = pd.read_csv('//home/deep/Desktop/NEWshoonya/MCX_symbols.txt.zip', compression='zip', engine='python', delimiter=',')
fno_scrips_mcx['Expiry'] = pd.to_datetime(fno_scrips_mcx['Expiry'])
fno_scrips_mcx['StrikePrice'] = fno_scrips_mcx['StrikePrice'].astype(float)
fno_scrips_mcx.sort_values('Expiry', inplace=True)
fno_scrips_mcx.reset_index(drop=True, inplace=True)

fno_scrips = pd.read_csv('/home/deep/Desktop/NEWshoonya/NFO_symbols.txt.zip', compression='zip', engine='python', delimiter=',')
fno_scrips['Expiry'] = pd.to_datetime(fno_scrips['Expiry'])
fno_scrips['StrikePrice'] = fno_scrips['StrikePrice'].astype(float)
fno_scrips.sort_values('Expiry', inplace=True)
fno_scrips.reset_index(drop=True, inplace=True)

In [ ]:
### VAr and global
feed_lock = threading.Lock()
resampling_lock = asyncio.Lock()
trade_execution_lock = threading.Lock()
range_length = 3
last_processed_candle = {}
trading_active = True
feed_opened = False
feedJson = defaultdict(lambda: deque(maxlen=10))  # Deque with maxlen=10 for each token
extra_feedJson = defaultdict(lambda: deque(maxlen=10))

last_resample_time = {}
resampled_data = {}
current_positions = {}
entry_instruments = {}
profit_target = 100
stop_loss = 20
trading_logic_task = None
option_symbols_subscribed = False

atm_strike: float
class TradingState:
    def __init__(self):
        self.ce_trading_symbol = None
        self.pe_trading_symbol = None
        self.ce_trading_token = None
        self.pe_trading_token = None

state = TradingState()

exchange = 'MCX'
symbol = 'CRUDEOILM'
Initial_token = '428870'
resample_frequency = '5s'

subscription_string = exchange + '|' + Initial_token

def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True

# Assuming `api` is defined and starts the WebSocket connection
async def connect_and_subscribe():
    global feed_opened
    api.start_websocket(
        order_update_callback=event_handler_order_update,
        subscribe_callback=event_handler_feed_update,
        socket_open_callback=open_callback
    )
    while not feed_opened:
        await asyncio.sleep(1)  # Wait for initial connection
    api.subscribe([subscription_string])


def set_trading_active(value):
    global trading_active, trading_logic_task
    trading_active = value
    print(f"Trading active set to {trading_active}")

    if trading_active:
        if trading_logic_task is None or trading_logic_task.done():
            trading_logic_task = asyncio.create_task(trading_logic())
            print("Started trading logic task")
    else:
        if trading_logic_task is not None and not trading_logic_task.done():
            trading_logic_task.cancel()
            print("Cancelled trading logic task")

def event_handler_feed_update(tick_data):
    #print(tick_data)
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']

            with feed_lock:
                if token == Initial_token:
                    # Append to the deque for the main token
                    feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
                
                else:
                    # Append to the deque for extra tokens
                    extra_feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})

    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")

async def resample_ticks():
    
    while True:
        if not feedJson:
            await asyncio.sleep(0.01)  # Sleep for a short interval if no data
            continue

        async with resampling_lock:
            for token, ticks in feedJson.items():
                if not ticks:
                    continue
                try:
                    # Create a DataFrame with the new ticks
                    df_new = pd.DataFrame(ticks)
                    df_new["tt"] = pd.to_datetime(df_new["tt"])
                    df_new.set_index("tt", inplace=True)

                    # Initialize or update the resampled DataFrame
                    if token not in resampled_data:
                        # Initialize the DataFrame with the first resample
                        resampled_data[token] = df_new['ltp'].resample(resample_frequency).ohlc()
                        last_resample_time[token] = df_new.index.max()
                    else:
                        # Resample the new ticks
                        df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()

                        #Update the existing resampled DataFrame with new data
                        # for idx, row in df_resampled.iterrows():
                        #     if idx in resampled_data[token].index:
                        #         resampled_data[token].at[idx, 'high'] = max(resampled_data[token].at[idx, 'high'], row['high'])
                        #         resampled_data[token].at[idx, 'low'] = min(resampled_data[token].at[idx, 'low'], row['low'])
                        #         resampled_data[token].at[idx, 'close'] = row['close']
                        #     else:
                        #         resampled_data[token].loc[idx] = row              

                        for idx, row in df_resampled.iterrows():
                            if idx in resampled_data[token].index:
                                # Update the existing row without triggering SettingWithCopyWarning
                                resampled_data[token].loc[idx, 'high'] = max(resampled_data[token].loc[idx, 'high'], row['high'])
                                resampled_data[token].loc[idx, 'low'] = min(resampled_data[token].loc[idx, 'low'], row['low'])
                                resampled_data[token].loc[idx, 'close'] = row['close']
                            else:
                                # Use the append method to avoid the warning
                                resampled_data[token] = resampled_data[token].append(row, ignore_index=False)                          
                                  

                        # Update the last resampled time
                        last_resample_time[token] = df_new.index.max()
                    
                    if token in resampled_data:
                        long_stop, short_stop, direction = numba_indicators.chandelier_exit_numba(resampled_data[token]['open'].values,resampled_data[token]['high'].values,resampled_data[token]['low'].values,resampled_data[token]['close'].values)

                        resampled_data[token]['long_stop'] = long_stop
                        resampled_data[token]['short_stop'] = short_stop
                        resampled_data[token]['ce_direction'] = direction
                        
                    resampled_data[token] = resampled_data[token].dropna(subset=['open', 'high', 'low', 'close'])

                except Exception as e:
                    if isinstance(e, KeyError) and e.args[0] == 'tt':
                        print(f"Market likely closed for token {token}")
                    else:
                        print(f"Error resampling data for token {token}: {e}")

        await asyncio.sleep(0.01)  # Sleep for a short interval to avoid busy-waiting


async def main():

    global trading_logic_task
   
    resample_task = asyncio.create_task(resample_ticks())
    connect_task = asyncio.create_task(connect_and_subscribe()) 
    trading_logic_task = asyncio.create_task(trading_logic())     
    #update_symbols_task = asyncio.create_task(update_atm_symbols())
    
    await asyncio.gather(connect_task, resample_task, trading_logic_task, update_symbols_task)

loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()

# Always create a task for 'main' within the current loop
asyncio.create_task(main())

# If the loop wasn't running, start it
if not loop.is_running():
    loop.run_forever() 


In [ ]:
async def trading_logic():
    
    global last_processed_candle, resampled_data, trading_active
    print("i trade_logic")
    while trading_active:
        current_time = pd.Timestamp.now()        
        
        async with resampling_lock:
            for token, data in resampled_data.items():
                if not data.empty:
                    last_candle_start = data.index[-1]
                    resample_freq = data.index.freq or pd.Timedelta(seconds=5)
                    current_candle_end = last_candle_start + resample_freq

                    # Check if the current time is beyond the current candle's end time
                    if current_time >= current_candle_end:
                        # Ensure we haven't processed this candle yet
                        
                        if token not in last_processed_candle or last_processed_candle[token] < current_candle_end:
                            last_processed_candle[token] = current_candle_end
                            
                            # Add the logic to process the resampled data here
                            print(f"[{pd.Timestamp.now()}] firing logic for: {token}")
                            await process_token_data(token)

            # Sleep briefly to yield control and avoid busy-waiting
            await asyncio.sleep(.5)  # Adjust the sleep duration as needed


In [ ]:
resampled_data

In [ ]:
async def process_token_data(token):
    async with resampling_lock:
        print("i am resample")
        df = resampled_data.get(token)

        if df is None or len(df) < 3:
            print(f"[{pd.Timestamp.now()}] Not enough data or token {token} not found. Need at least 3 candles.")
            return
        try:
            # Get the current time
            current_time = pd.Timestamp.now()
            # Get the end of the last completed candle 
            new_candle_start_time = current_time.floor(resample_frequency)

            # Calculate the start of the last completed candle
            resample_freq_timedelta = pd.Timedelta(resample_frequency)
            last_completed_candle_start_time = new_candle_start_time - resample_freq_timedelta

            # Filter completed candles
            if new_candle_start_time in df.index:
                completed_candles_df = df.loc[:new_candle_start_time - pd.Timedelta(microseconds=1)] 
            elif last_completed_candle_start_time in df.index:
                completed_candles_df = df

            try:
                # Check if expected columns exist
                required_columns = ['ce_direction'] 
                missing_columns = [col for col in required_columns if col not in completed_candles_df.columns]
                if missing_columns:
                    raise KeyError(f"Missing columns {missing_columns}")

                # Extract current and previous directions
                current_direction = completed_candles_df['ce_direction'].iloc[-1]
                previous_direction = completed_candles_df['ce_direction'].iloc[-2]

                print(f"[{pd.Timestamp.now()}] Token: {token}, Current Direction: {current_direction}, Previous Direction: {previous_direction}")
                # Execute trade logic
                await execute_trade_logic(token, current_direction, previous_direction, completed_candles_df) 

            except KeyError as e:
                print(f"[{pd.Timestamp.now()}] Error processing token {token}: {e}") 

        except KeyError as e:
            print(f"[{pd.Timestamp.now()}] Error processing token {token}: Column {e} not found.") 
        except IndexError:
            print(f"[{pd.Timestamp.now()}] Error processing token {token}: Not enough candles to calculate indicators.")


In [ ]:
async def execute_trade_logic(token, current_direction, previous_direction, completed_candles_df):
    global current_positions, trade_execution_lock
    print("i am execute_trade_logic")

    with trade_execution_lock:  
        # Exit Logic (Prioritized)
        if token in current_positions:
            position_data = current_positions[token]
            op_token = position_data["option_token"]
            op_symbol = position_data["symbol_name"]
            position_type = position_data["position_type"]

            if (position_type == 'call_buy' and current_direction == -1 and previous_direction == 1) or \
               (position_type == 'put_buy' and current_direction == 1 and previous_direction == -1):
                
                try:
                    await asyncio.to_thread(exit_trade(token, position_type, op_token, op_symbol))
                except Exception as e:
                    print(f"Error exiting trade for {token}: {e}")

        # Entry Logic 
        if token not in current_positions:  
            if current_direction == 1 and previous_direction == -1: 
                position_type = "call_buy"
            elif current_direction == -1 and previous_direction == 1:
                position_type = "put_buy"
            else:
                return  # No entry signal

            try:
                await asyncio.to_thread(enter_trade(token, position_type))                
            except Exception as e:
                print(f"Error entering trade for {token}: {e}")


In [ ]:
def enter_trade(token, position_type):
    with trade_execution_lock:
        with trade_execution_lock:
            option_token = state.ce_trading_token if position_type == "call_buy" else state.pe_trading_token
            symbol_name = state.ce_trading_symbol if position_type == "call_buy" else state.pe_trading_symbol
            timestamp - resampled_data.index[-1]
            current_positions[token] = {
                        "position_type": position_type,
                        "option_token": option_token,
                        "symbol_name": symbol_name
                    }
            action_time = pd.Timestamp.now()
            log_trade(timestamp, symbol_name, "ENTRY", price, action_time) 
            print(f"[{time.strftime('%H:%M:%S.%f', time.localtime(time.time()))[:-3]}] Entered {position_type} trade for {symbol_name} at {action_time} for dataframe index time {timestamp}")


def exit_trade(token, position_type, op_token, op_symbol):
    with trade_execution_lock: 
        price = get_latest_price_option(op_token)
        action_time = pd.Timestamp.now()
        timestamp - resampled_data.index[-1]
        log_trade(timestamp, op_symbol, "EXIT", price, action_time)
        print(f"[{time.strftime('%H:%M:%S.%f', time.localtime(time.time()))[:-3]}] Exited {position_type} trade for {op_symbol} at {action_time} for dataframe index time {timestamp}")
        del current_positions[token]  # Remove the entire token entry
        print ("i am out trade")


In [ ]:
def get_latest_price_option(entry_instrument):
    entry_instrument_str = str(entry_instrument)
    with feed_lock:
        if entry_instrument_str in extra_feedJson:
            latest_data = extra_feedJson[entry_instrument_str][-1]
            latest_price = latest_data['ltp']
            
            return latest_price
        else:
            print(f"{entry_instrument_str} not found in extra_feedJson")  # Debug print
            return None
        
def get_latest_price_index(entry_instrument):
    entry_instrument_str = str(entry_instrument)
    with feed_lock:
        if entry_instrument_str in feedJson:
            latest_data = feedJson[entry_instrument_str][-1]
            latest_price = latest_data['ltp']
            
            return latest_price
        else:
            print(f"{entry_instrument_str} not found in feedJson")  # Debug print
            return None

In [ ]:
# Define the function
def find_atm_strike_and_symbols():
    global atm_strike, filtered_df
    cmp_bn = float(resampled_data[Initial_token]['close'].iloc[-1])
    #cmp_bn = '50270'
    if pd.isna(cmp_bn):
        return None, None, None, None

    cmp_bn = int(float(cmp_bn))
    mod = cmp_bn % 100
    atm_strike = cmp_bn - mod if mod <= 50 else cmp_bn + (100 - mod)

    if exchange == 'NSE':
        filtered_df = fno_scrips[
                (fno_scrips['Symbol'] == symbol) &
                (fno_scrips['StrikePrice'] == float(atm_strike))
            ]
    elif exchange == 'MCX':
        filtered_df = fno_scrips_mcx[
            (fno_scrips_mcx['Symbol'] == symbol) &
            (fno_scrips_mcx['StrikePrice'] == float(atm_strike))
        ]

    if filtered_df.empty:
        print(f"Could not find trading symbols for ATM strike {atm_strike}")
        return None, None, None, None

    ce_row = filtered_df[filtered_df['OptionType'] == 'CE'].sort_values('Expiry').iloc[0]
    pe_row = filtered_df[filtered_df['OptionType'] == 'PE'].sort_values('Expiry').iloc[0]

    ce_trading_symbol, pe_trading_symbol = ce_row['TradingSymbol'], pe_row['TradingSymbol']
    ce_trading_token, pe_trading_token = ce_row['Token'], pe_row['Token']

    return ce_trading_symbol, pe_trading_symbol, ce_trading_token, pe_trading_token

# Update ATM symbols function
# async def update_atm_symbols():
def update_atm_symbols():
    global option_symbols_subscribed, atm_strike
    try:
        # while True:
            result = find_atm_strike_and_symbols() # Correct way to call the function

            if result[0] is not None:
                (state.ce_trading_symbol, state.pe_trading_symbol,
                state.ce_trading_token, state.pe_trading_token) = result

                # if not option_symbols_subscribed:
                #     option_subscription()  # Implement this function
                #     option_symbols_subscribed = True
            else:
                print("Failed to update symbols")

            time.sleep(10)

    except Exception as e:
        print(f"Error in update_atm_symbols: {e}")



In [ ]:
def option_subscription():
    op_chain = api.get_option_chain(exchange='NFO', tradingsymbol=state.ce_trading_symbol, strikeprice=atm_strike, count =2)
    tokens = [item['token'] for item in op_chain['values']]
    subscriptions = [f"NFO|{token}" for token in tokens]
    # Subscribe to each token
    for sub in subscriptions:
        api.subscribe(sub)

    # Print the subscriptions for verification
    print("Subscribed to:", subscriptions)